# 🎵 SonicSentinel AI — Model Training Notebook
**Project:** SonicSentinel AI | **Theme:** AcousticX Intelligence | **Category:** NextWave AI & ML

This notebook covers the complete ML pipeline:
1. Setup & Dependencies
2. Dataset Loading & EDA
3. Audio Preprocessing
4. Data Augmentation
5. Feature Extraction (MFCC, Mel Spectrogram, Chroma, etc.)
6. Model Training (SVM, Random Forest, CNN)
7. Hyperparameter Tuning
8. Model Evaluation & Comparison
9. Confusion Matrices & Classification Reports
10. Model Export

## 📦 Step 1: Install Dependencies

In [ ]:
# Install required packages
!pip install librosa soundfile scikit-learn seaborn pandas numpy matplotlib joblib audioread pydub -q

# Install TensorFlow for CNN (GPU accelerated on Colab)
!pip install tensorflow -q

# Optional: XGBoost
!pip install xgboost -q

print('✓ All packages installed')

## 📁 Step 2: Mount Google Drive & Load Dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ─────────────────────────────────────────────
# UPDATE THIS PATH to your Drive folder
# ─────────────────────────────────────────────
DATA_DIR = '/content/drive/MyDrive/SonicSentinel/audio data'  # <-- Change this

import os
from pathlib import Path

# Verify dataset exists
data_path = Path(DATA_DIR)
if data_path.exists():
    print(f'✓ Dataset found at: {DATA_DIR}')
    for cls in sorted(data_path.iterdir()):
        if cls.is_dir():
            count = len(list(cls.glob('*.wav')) + list(cls.glob('*.mp3')))
            print(f'  {cls.name}: {count} files')
else:
    print(f'✗ Dataset not found at {DATA_DIR}')
    print('Please upload your audio data folder to Google Drive')

## ⚙️ Step 3: Configuration & Constants

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import librosa
import librosa.display
import soundfile as sf
import warnings
import json
import pickle
import time
import uuid
from pathlib import Path
from IPython.display import Audio, display, clear_output

warnings.filterwarnings('ignore')
matplotlib.rcParams['figure.dpi'] = 100

# ─────────────────────────────────────────────
# Constants
# ─────────────────────────────────────────────
SAMPLE_RATE   = 22050
N_MFCC        = 40
N_MELS        = 128
HOP_LENGTH    = 512
N_FFT         = 2048
SEGMENT_DUR   = 3.0    # seconds per segment
RANDOM_SEED   = 42
CV_FOLDS      = 5

SOUND_CLASSES = [
    'aggression', 'alarm_siren', 'animal_sound', 'background_noise',
    'glass_breaking', 'gunshot', 'machinery_fault',
    'panic_scream', 'person_asking_for_help', 'vehicle_horn'
]
NUM_CLASSES = len(SOUND_CLASSES)

CLASS_LABELS = {
    'aggression': 'Aggression', 'alarm_siren': 'Alarm/Siren',
    'animal_sound': 'Animal Sound', 'background_noise': 'Background Noise',
    'glass_breaking': 'Glass Breaking', 'gunshot': 'Gunshot',
    'machinery_fault': 'Machinery Fault', 'panic_scream': 'Panic Scream',
    'person_asking_for_help': 'Person Asking Help', 'vehicle_horn': 'Vehicle Horn'
}

CRITICAL_CLASSES = {'gunshot', 'panic_scream', 'person_asking_for_help', 'glass_breaking', 'aggression'}

OUTPUT_DIR = '/content/sonic_sentinel_models'
Path(OUTPUT_DIR).mkdir(exist_ok=True)

np.random.seed(RANDOM_SEED)
print(f'✓ Configuration set | {NUM_CLASSES} classes | SR={SAMPLE_RATE}Hz')
print(f'Output directory: {OUTPUT_DIR}')

## 🔍 Step 4: Exploratory Data Analysis (EDA)

In [ ]:
# ─────────────────────────────────────────────
# Scan dataset
# ─────────────────────────────────────────────
records = []
for cls in SOUND_CLASSES:
    cls_dir = Path(DATA_DIR) / cls
    if not cls_dir.exists():
        print(f'Warning: {cls_dir} not found')
        continue
    files = list(cls_dir.glob('*.wav')) + list(cls_dir.glob('*.mp3'))
    for fp in files:
        try:
            info = sf.info(str(fp))
            records.append({
                'audio_id': f'AUD-{uuid.uuid4().hex[:8]}',
                'filename': fp.name,
                'filepath': str(fp),
                'class_label': cls,
                'duration': info.duration,
                'sample_rate': info.samplerate,
                'channels': info.channels,
            })
        except Exception as e:
            print(f'Cannot read {fp.name}: {e}')

df_manifest = pd.DataFrame(records)
print(f'Total files: {len(df_manifest)}')
print(f'\nClass distribution:')
print(df_manifest['class_label'].value_counts())

In [ ]:
# Visualize class distribution
fig, axes = plt.subplots(1, 2, figsize=(16, 5), facecolor='#0a0e1a')
for ax in axes: ax.set_facecolor('#0d1526')

# Bar chart
counts = df_manifest['class_label'].value_counts()
colors = ['#00d4ff' if c in CRITICAL_CLASSES else '#1e5f8c' for c in counts.index]
axes[0].barh([CLASS_LABELS.get(c, c) for c in counts.index], counts.values,
             color=colors, edgecolor='none')
axes[0].set_xlabel('File Count', color='#94a3b8')
axes[0].set_title('Class Distribution (cyan = critical)', color='#e2e8f0', fontsize=12)
axes[0].tick_params(colors='#94a3b8')
axes[0].spines[:].set_color('#1e2d4a')

# Duration distribution
axes[1].hist(df_manifest['duration'], bins=50, color='#7c3aed', edgecolor='none', alpha=0.8)
axes[1].set_xlabel('Duration (seconds)', color='#94a3b8')
axes[1].set_ylabel('Count', color='#94a3b8')
axes[1].set_title('Audio Duration Distribution', color='#e2e8f0', fontsize=12)
axes[1].tick_params(colors='#94a3b8')
axes[1].spines[:].set_color('#1e2d4a')

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/eda_distribution.png', dpi=120, facecolor='#0a0e1a')
plt.show()
print(f'Duration stats:\n{df_manifest["duration"].describe()}')

In [ ]:
# Listen to a sample from each class
print('Sample audio from each class:')
for cls in SOUND_CLASSES[:3]:  # Show first 3 to keep output concise
    sample = df_manifest[df_manifest['class_label'] == cls].iloc[0]
    y, sr = librosa.load(sample['filepath'], sr=SAMPLE_RATE, mono=True, duration=3.0)
    print(f'  {CLASS_LABELS[cls]}:')
    display(Audio(y, rate=sr))

## 🔧 Step 5: Audio Preprocessing Functions

In [ ]:
def load_and_preprocess(filepath, sr=SAMPLE_RATE, duration=SEGMENT_DUR,
                         apply_noise_reduction=True):
    """
    Load audio file, apply preprocessing pipeline:
    - Resample to target SR
    - Convert to mono
    - Trim silence
    - Normalize amplitude
    - Apply noise reduction
    - Pad or truncate to fixed duration
    """
    # Load
    y, _ = librosa.load(filepath, sr=sr, mono=True)

    # Trim silence
    y, _ = librosa.effects.trim(y, top_db=30)

    # Normalize
    max_val = np.max(np.abs(y))
    if max_val > 0:
        y = y / max_val

    # Simple spectral gating noise reduction
    if apply_noise_reduction and len(y) > 100:
        noise_samples = min(int(0.3 * sr), len(y) // 4)
        if noise_samples > 50:
            stft = librosa.stft(y, n_fft=N_FFT, hop_length=HOP_LENGTH)
            mag, phase = np.abs(stft), np.angle(stft)
            noise_mag = np.mean(mag[:, :max(1, noise_samples // HOP_LENGTH)], axis=1, keepdims=True)
            mask = mag > (noise_mag * 1.5)
            clean_stft = mag * mask * np.exp(1j * phase)
            y = librosa.istft(clean_stft, hop_length=HOP_LENGTH, length=len(y))

    # Pad or truncate to fixed duration
    target_len = int(duration * sr)
    if len(y) < target_len:
        y = np.pad(y, (0, target_len - len(y)), mode='constant')
    else:
        y = y[:target_len]

    return y.astype(np.float32)


def assess_quality(y, sr=SAMPLE_RATE):
    """Assess audio quality: returns quality tier and metrics."""
    rms = float(np.sqrt(np.mean(y**2)))
    clip_ratio = float(np.mean(np.abs(y) >= 0.99))
    noise_floor = np.percentile(np.abs(y), 10)
    signal_peak = np.percentile(np.abs(y), 90)
    snr_db = float(20 * np.log10((signal_peak + 1e-9) / (noise_floor + 1e-9)))

    if rms > 0.05 and clip_ratio < 0.01 and snr_db > 20:
        quality = 'good'
    elif rms > 0.02 and clip_ratio < 0.05 and snr_db > 10:
        quality = 'acceptable'
    elif rms > 0.005 and clip_ratio < 0.15 and snr_db > 5:
        quality = 'poor'
    else:
        quality = 'unusable'

    return {'quality': quality, 'rms': rms, 'clip_ratio': clip_ratio, 'snr_db': snr_db}


# Test preprocessing
test_file = df_manifest.iloc[0]
y_test = load_and_preprocess(test_file['filepath'])
quality = assess_quality(y_test)
print(f'Test preprocessing on: {test_file["filename"]}')
print(f'  Output shape: {y_test.shape} ({len(y_test)/SAMPLE_RATE:.2f}s)')
print(f'  Quality: {quality}')

In [ ]:
# Visualize waveform and spectrogram
fig, axes = plt.subplots(2, 2, figsize=(16, 8), facecolor='#0a0e1a')
for ax in axes.flat: ax.set_facecolor('#0d1526')

samples_to_show = [
    df_manifest[df_manifest['class_label'] == 'gunshot'].iloc[0],
    df_manifest[df_manifest['class_label'] == 'glass_breaking'].iloc[0],
]

for col, sample in enumerate(samples_to_show):
    y_s = load_and_preprocess(sample['filepath'])
    cls_label = CLASS_LABELS[sample['class_label']]

    # Waveform
    t = np.linspace(0, SEGMENT_DUR, len(y_s))
    axes[0][col].plot(t, y_s, color='#00d4ff', linewidth=0.5, alpha=0.8)
    axes[0][col].fill_between(t, y_s, alpha=0.3, color='#00d4ff')
    axes[0][col].set_title(f'Waveform: {cls_label}', color='#e2e8f0', fontsize=11)
    axes[0][col].set_xlabel('Time (s)', color='#94a3b8')
    axes[0][col].tick_params(colors='#94a3b8')
    axes[0][col].spines[:].set_color('#1e2d4a')

    # Mel Spectrogram
    mel = librosa.feature.melspectrogram(y=y_s, sr=SAMPLE_RATE, n_mels=N_MELS,
                                          n_fft=N_FFT, hop_length=HOP_LENGTH)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    img = librosa.display.specshow(mel_db, x_axis='time', y_axis='mel',
                                    sr=SAMPLE_RATE, hop_length=HOP_LENGTH,
                                    cmap='magma', ax=axes[1][col])
    axes[1][col].set_title(f'Mel Spectrogram: {cls_label}', color='#e2e8f0', fontsize=11)
    axes[1][col].tick_params(colors='#94a3b8')
    axes[1][col].spines[:].set_color('#1e2d4a')
    plt.colorbar(img, ax=axes[1][col], format='%+2.0f dB').ax.yaxis.set_tick_params(color='#94a3b8')

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/waveform_spectrogram_samples.png', dpi=120, facecolor='#0a0e1a')
plt.show()

## 🎭 Step 6: Data Augmentation

In [ ]:
import random
random.seed(RANDOM_SEED)

def augment_audio(y, sr=SAMPLE_RATE, p=0.5):
    """
    Apply random augmentations to audio:
    - Gaussian noise addition
    - Time shifting
    - Pitch shifting  
    - Time stretching
    - Volume variation
    - Reverberation simulation
    """
    y = y.copy().astype(np.float32)

    # 1. Gaussian noise
    if random.random() < p:
        noise_factor = random.uniform(0.003, 0.015)
        y = y + np.random.randn(len(y)).astype(np.float32) * noise_factor

    # 2. Time shift
    if random.random() < p:
        shift = random.uniform(-0.25, 0.25)
        shift_samples = int(len(y) * shift)
        if shift_samples > 0:
            y = np.pad(y[:-shift_samples], (shift_samples, 0), mode='constant')
        elif shift_samples < 0:
            y = np.pad(y[-shift_samples:], (0, -shift_samples), mode='constant')

    # 3. Pitch shift
    if random.random() < p * 0.7:  # Less frequent (computationally expensive)
        n_steps = random.uniform(-2.5, 2.5)
        try:
            y = librosa.effects.pitch_shift(y, sr=sr, n_steps=n_steps)
        except Exception:
            pass

    # 4. Time stretch
    if random.random() < p * 0.7:
        rate = random.uniform(0.85, 1.2)
        try:
            y_stretched = librosa.effects.time_stretch(y, rate=rate)
            target_len = len(y)
            if len(y_stretched) < target_len:
                y = np.pad(y_stretched, (0, target_len - len(y_stretched)))
            else:
                y = y_stretched[:target_len]
        except Exception:
            pass

    # 5. Volume variation
    if random.random() < p:
        factor = random.uniform(0.6, 1.4)
        y = np.clip(y * factor, -1.0, 1.0)

    # 6. Simple reverb
    if random.random() < p * 0.4:
        ir_len = int(0.05 * sr)
        decay = np.exp(-6 * np.linspace(0, 1, ir_len))
        ir = decay * np.random.randn(ir_len) * 0.15
        ir[0] = 1.0
        try:
            y_rev = np.convolve(y, ir, mode='full')[:len(y)]
            max_v = np.max(np.abs(y_rev))
            if max_v > 0:
                y_rev = y_rev / max_v * np.max(np.abs(y))
            y = y_rev.astype(np.float32)
        except Exception:
            pass

    # Normalize
    max_v = np.max(np.abs(y))
    if max_v > 0:
        y = y / max_v

    return y.astype(np.float32)


# Demonstrate augmentation
sample = df_manifest[df_manifest['class_label'] == 'gunshot'].iloc[0]
y_orig = load_and_preprocess(sample['filepath'])
y_aug1 = augment_audio(y_orig, p=0.8)
y_aug2 = augment_audio(y_orig, p=0.8)

fig, axes = plt.subplots(3, 1, figsize=(14, 7), facecolor='#0a0e1a')
for ax in axes: ax.set_facecolor('#0d1526')
t = np.linspace(0, SEGMENT_DUR, len(y_orig))
titles = ['Original', 'Augmented (v1)', 'Augmented (v2)']
colors = ['#00d4ff', '#ff6b35', '#a855f7']
for i, (y, title, color) in enumerate(zip([y_orig, y_aug1, y_aug2], titles, colors)):
    axes[i].plot(t, y, color=color, linewidth=0.6, alpha=0.85)
    axes[i].set_title(title, color='#e2e8f0', fontsize=10)
    axes[i].tick_params(colors='#94a3b8')
    axes[i].spines[:].set_color('#1e2d4a')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/augmentation_demo.png', dpi=120, facecolor='#0a0e1a')
plt.show()
print('✓ Augmentation pipeline demonstrated')

## 🧪 Step 7: Feature Extraction

In [ ]:
def extract_features(y, sr=SAMPLE_RATE):
    """
    Extract comprehensive acoustic feature vector:
    - MFCC (40 coefficients) + delta + delta-delta: 240
    - Chroma STFT + CQT: 48
    - Zero Crossing Rate: 4
    - RMS Energy: 4
    - Spectral Centroid: 3
    - Spectral Bandwidth: 3
    - Spectral Roll-off (85%, 95%): 4
    - Spectral Contrast: 14
    - Spectral Flatness: 2
    - Onset Strength: 4
    - Tempo: 1
    - Tonnetz: 12
    Total: ~339 features
    """
    features = []

    # MFCC + deltas
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=N_MFCC, n_fft=N_FFT, hop_length=HOP_LENGTH)
    mfcc_d = librosa.feature.delta(mfcc)
    mfcc_d2 = librosa.feature.delta(mfcc, order=2)
    for m in [mfcc, mfcc_d, mfcc_d2]:
        features.extend(np.mean(m, axis=1))
        features.extend(np.std(m, axis=1))

    # Chroma STFT
    chroma = librosa.feature.chroma_stft(y=y, sr=sr, n_fft=N_FFT, hop_length=HOP_LENGTH)
    features.extend(np.mean(chroma, axis=1))
    features.extend(np.std(chroma, axis=1))

    # Chroma CQT
    try:
        chroma_cqt = librosa.feature.chroma_cqt(y=y, sr=sr, hop_length=HOP_LENGTH)
        features.extend(np.mean(chroma_cqt, axis=1))
        features.extend(np.std(chroma_cqt, axis=1))
    except Exception:
        features.extend(np.zeros(24))

    # ZCR
    zcr = librosa.feature.zero_crossing_rate(y, frame_length=N_FFT, hop_length=HOP_LENGTH)
    features.extend([np.mean(zcr), np.std(zcr), np.max(zcr), np.min(zcr)])

    # RMS
    rms = librosa.feature.rms(y=y, frame_length=N_FFT, hop_length=HOP_LENGTH)
    features.extend([np.mean(rms), np.std(rms), np.max(rms), np.min(rms)])

    # Spectral Centroid
    sc = librosa.feature.spectral_centroid(y=y, sr=sr, n_fft=N_FFT, hop_length=HOP_LENGTH)
    features.extend([np.mean(sc), np.std(sc), np.max(sc)])

    # Spectral Bandwidth
    bw = librosa.feature.spectral_bandwidth(y=y, sr=sr, n_fft=N_FFT, hop_length=HOP_LENGTH)
    features.extend([np.mean(bw), np.std(bw), np.max(bw)])

    # Spectral Rolloff
    for roll in [0.85, 0.95]:
        ro = librosa.feature.spectral_rolloff(y=y, sr=sr, roll_percent=roll,
                                               n_fft=N_FFT, hop_length=HOP_LENGTH)
        features.extend([np.mean(ro), np.std(ro)])

    # Spectral Contrast
    try:
        sc2 = librosa.feature.spectral_contrast(y=y, sr=sr, n_fft=N_FFT, hop_length=HOP_LENGTH)
        features.extend(np.mean(sc2, axis=1))
        features.extend(np.std(sc2, axis=1))
    except Exception:
        features.extend(np.zeros(14))

    # Spectral Flatness
    sf2 = librosa.feature.spectral_flatness(y=y, n_fft=N_FFT, hop_length=HOP_LENGTH)
    features.extend([np.mean(sf2), np.std(sf2)])

    # Onset Strength
    onset_env = librosa.onset.onset_strength(y=y, sr=sr, hop_length=HOP_LENGTH)
    features.extend([np.mean(onset_env), np.std(onset_env),
                     np.max(onset_env), np.sum(onset_env > np.mean(onset_env))])

    # Tempo
    try:
        tempo, _ = librosa.beat.beat_track(y=y, sr=sr, hop_length=HOP_LENGTH)
        features.append(float(tempo))
    except Exception:
        features.append(0.0)

    # Tonnetz
    try:
        y_harm = librosa.effects.harmonic(y)
        tonnetz = librosa.feature.tonnetz(y=y_harm, sr=sr)
        features.extend(np.mean(tonnetz, axis=1))
        features.extend(np.std(tonnetz, axis=1))
    except Exception:
        features.extend(np.zeros(12))

    features = np.array(features, dtype=np.float32)
    return np.nan_to_num(features, nan=0.0, posinf=0.0, neginf=0.0)


# Test feature extraction
test_feats = extract_features(y_orig)
print(f'Feature vector shape: {test_feats.shape}')
print(f'Feature range: [{test_feats.min():.3f}, {test_feats.max():.3f}]')
print(f'NaN count: {np.isnan(test_feats).sum()}')

## 📊 Step 8: Build Feature Dataset

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Stratified split BEFORE feature extraction (to prevent data leakage)
from sklearn.model_selection import train_test_split

X_paths = df_manifest['filepath'].values
y_labels = df_manifest['class_label'].values

X_train_p, X_temp_p, y_train_l, y_temp_l = train_test_split(
    X_paths, y_labels, test_size=0.30, random_state=RANDOM_SEED, stratify=y_labels
)
X_val_p, X_test_p, y_val_l, y_test_l = train_test_split(
    X_temp_p, y_temp_l, test_size=0.50, random_state=RANDOM_SEED, stratify=y_temp_l
)

print(f'Train: {len(X_train_p)} | Val: {len(X_val_p)} | Test: {len(X_test_p)}')
print(f'Train class distribution: {pd.Series(y_train_l).value_counts().to_dict()}')

In [ ]:
def build_feature_set(paths, labels, split_name, augment=False, aug_per_sample=2):
    """Extract features from all files in a split."""
    X_list, y_list = [], []
    skipped = 0
    total = len(paths)

    for i, (fp, label) in enumerate(zip(paths, labels)):
        if i % 100 == 0:
            print(f'  [{split_name}] {i}/{total} ({label})', end='\r')
        try:
            y_audio = load_and_preprocess(fp)
            feats = extract_features(y_audio)
            X_list.append(feats)
            y_list.append(label)

            # Augmentation (only for training)
            if augment:
                for _ in range(aug_per_sample):
                    y_aug = augment_audio(y_audio, p=0.5)
                    feats_aug = extract_features(y_aug)
                    X_list.append(feats_aug)
                    y_list.append(label)

        except Exception as e:
            skipped += 1

    print(f'  [{split_name}] Done: {len(X_list)} features | Skipped: {skipped}          ')
    return np.array(X_list, dtype=np.float32), np.array(y_list)


print('Extracting training features (with augmentation)...')
X_train, y_train = build_feature_set(X_train_p, y_train_l, 'train', augment=True, aug_per_sample=2)

print('Extracting validation features...')
X_val, y_val = build_feature_set(X_val_p, y_val_l, 'val', augment=False)

print('Extracting test features...')
X_test, y_test = build_feature_set(X_test_p, y_test_l, 'test', augment=False)

print(f'\nDataset Summary:')
print(f'  X_train: {X_train.shape} | y_train: {y_train.shape}')
print(f'  X_val:   {X_val.shape} | y_val:   {y_val.shape}')
print(f'  X_test:  {X_test.shape} | y_test:  {y_test.shape}')

# Save numpy arrays
np.save(f'{OUTPUT_DIR}/X_train.npy', X_train)
np.save(f'{OUTPUT_DIR}/y_train.npy', y_train)
np.save(f'{OUTPUT_DIR}/X_val.npy', X_val)
np.save(f'{OUTPUT_DIR}/y_val.npy', y_val)
np.save(f'{OUTPUT_DIR}/X_test.npy', X_test)
np.save(f'{OUTPUT_DIR}/y_test.npy', y_test)
print('✓ Feature arrays saved')

In [ ]:
from sklearn.preprocessing import StandardScaler, LabelEncoder

# Encode labels
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_val_enc   = le.transform(y_val)
y_test_enc  = le.transform(y_test)

# Scale features (fit ONLY on train)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_val_sc   = scaler.transform(X_val)
X_test_sc  = scaler.transform(X_test)

# Save scaler and encoder
with open(f'{OUTPUT_DIR}/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
with open(f'{OUTPUT_DIR}/label_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)

print(f'Classes: {list(le.classes_)}')
print(f'X_train scaled shape: {X_train_sc.shape}')
print(f'Scaler mean range: [{scaler.mean_.min():.3f}, {scaler.mean_.max():.3f}]')

## 🤖 Step 9: Train Model 1 — Support Vector Machine (SVM)

In [ ]:
from sklearn.svm import SVC
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, cross_val_score
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                              f1_score, recall_score)

print('='*60)
print('Training SVM with Hyperparameter Tuning...')
print('='*60)

# Hyperparameter search space
svm_param_grid = {
    'C': [1, 5, 10, 50, 100],
    'kernel': ['rbf', 'poly'],
    'gamma': ['scale', 'auto'],
    'degree': [2, 3],  # For poly kernel
}

svm_base = SVC(
    probability=True,
    class_weight='balanced',
    random_state=RANDOM_SEED,
    max_iter=5000
)

svm_search = RandomizedSearchCV(
    svm_base,
    svm_param_grid,
    n_iter=15,
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_SEED),
    scoring='f1_macro',
    n_jobs=-1,
    random_state=RANDOM_SEED,
    verbose=1,
)

t0 = time.time()
svm_search.fit(X_train_sc, y_train_enc)
svm_train_time = time.time() - t0

best_svm = svm_search.best_estimator_
print(f'\n✓ Best SVM params: {svm_search.best_params_}')
print(f'  CV F1-macro: {svm_search.best_score_:.4f}')
print(f'  Training time: {svm_train_time:.1f}s')

# Save SVM
with open(f'{OUTPUT_DIR}/svm_model.pkl', 'wb') as f:
    pickle.dump(best_svm, f)
print('✓ SVM saved')

In [ ]:
# Evaluate SVM
y_pred_svm = best_svm.predict(X_test_sc)
svm_acc = accuracy_score(y_test_enc, y_pred_svm)
svm_f1  = f1_score(y_test_enc, y_pred_svm, average='macro', zero_division=0)
svm_f1w = f1_score(y_test_enc, y_pred_svm, average='weighted', zero_division=0)

print(f'\nSVM Test Results:')
print(f'  Accuracy:          {svm_acc:.4f} ({svm_acc*100:.2f}%)')
print(f'  F1-Score (macro):  {svm_f1:.4f}')
print(f'  F1-Score (wt):     {svm_f1w:.4f}')
print(f'\nClassification Report:')
print(classification_report(y_test_enc, y_pred_svm,
                             target_names=le.classes_, zero_division=0))

# Critical class recall
critical_recalls_svm = {}
for cls in CRITICAL_CLASSES:
    if cls in le.classes_:
        cls_idx = list(le.classes_).index(cls)
        rec = recall_score((y_test_enc == cls_idx).astype(int),
                           (y_pred_svm == cls_idx).astype(int), zero_division=0)
        critical_recalls_svm[cls] = rec
        status = '✓' if rec >= 0.85 else '✗'
        print(f'{status} {CLASS_LABELS.get(cls, cls)}: {rec:.4f}')

## 🌳 Step 10: Train Model 2 — Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

print('='*60)
print('Training Random Forest with Hyperparameter Tuning...')
print('='*60)

rf_param_dist = {
    'n_estimators': [200, 300, 500, 800],
    'max_depth': [None, 20, 30, 40],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
    'max_features': ['sqrt', 'log2'],
    'bootstrap': [True, False],
}

rf_base = RandomForestClassifier(
    class_weight='balanced',
    n_jobs=-1,
    random_state=RANDOM_SEED
)

rf_search = RandomizedSearchCV(
    rf_base,
    rf_param_dist,
    n_iter=20,
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_SEED),
    scoring='f1_macro',
    n_jobs=-1,
    random_state=RANDOM_SEED,
    verbose=1,
)

t0 = time.time()
rf_search.fit(X_train_sc, y_train_enc)
rf_train_time = time.time() - t0

best_rf = rf_search.best_estimator_
print(f'\n✓ Best RF params: {rf_search.best_params_}')
print(f'  CV F1-macro: {rf_search.best_score_:.4f}')
print(f'  Training time: {rf_train_time:.1f}s')

with open(f'{OUTPUT_DIR}/rf_model.pkl', 'wb') as f:
    pickle.dump(best_rf, f)
print('✓ Random Forest saved')

In [ ]:
# Evaluate RF
y_pred_rf = best_rf.predict(X_test_sc)
rf_acc = accuracy_score(y_test_enc, y_pred_rf)
rf_f1  = f1_score(y_test_enc, y_pred_rf, average='macro', zero_division=0)
rf_f1w = f1_score(y_test_enc, y_pred_rf, average='weighted', zero_division=0)

print(f'\nRandom Forest Test Results:')
print(f'  Accuracy:          {rf_acc:.4f} ({rf_acc*100:.2f}%)')
print(f'  F1-Score (macro):  {rf_f1:.4f}')
print(f'  F1-Score (wt):     {rf_f1w:.4f}')
print(f'\nClassification Report:')
print(classification_report(y_test_enc, y_pred_rf,
                             target_names=le.classes_, zero_division=0))

# Critical class recall
critical_recalls_rf = {}
for cls in CRITICAL_CLASSES:
    if cls in le.classes_:
        cls_idx = list(le.classes_).index(cls)
        rec = recall_score((y_test_enc == cls_idx).astype(int),
                           (y_pred_rf == cls_idx).astype(int), zero_division=0)
        critical_recalls_rf[cls] = rec
        status = '✓' if rec >= 0.85 else '✗'
        print(f'{status} {CLASS_LABELS.get(cls, cls)}: {rec:.4f}')

# Feature importance plot
importances = best_rf.feature_importances_
top_idx = np.argsort(importances)[-20:]
fig, ax = plt.subplots(figsize=(10, 6), facecolor='#0a0e1a')
ax.set_facecolor('#0d1526')
ax.barh(range(len(top_idx)), importances[top_idx], color='#00d4ff')
ax.set_title('Top 20 Feature Importances (Random Forest)', color='#e2e8f0')
ax.set_xlabel('Importance', color='#94a3b8')
ax.tick_params(colors='#94a3b8')
ax.spines[:].set_color('#1e2d4a')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/rf_feature_importance.png', dpi=120, facecolor='#0a0e1a')
plt.show()

## 🧠 Step 11: Train Model 3 — CNN (Convolutional Neural Network)

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from sklearn.utils.class_weight import compute_class_weight

print(f'TensorFlow version: {tf.__version__}')
print(f'GPU available: {len(tf.config.list_physical_devices("GPU")) > 0}')
tf.random.set_seed(RANDOM_SEED)

# ─────────────────────────────────────────────
# Build Mel spectrogram dataset for CNN
# ─────────────────────────────────────────────
TIME_FRAMES = int(np.ceil(SEGMENT_DUR * SAMPLE_RATE / HOP_LENGTH)) + 1

def extract_mel_spec(y, sr=SAMPLE_RATE, n_mels=N_MELS, fixed_time=TIME_FRAMES):
    """Extract fixed-size 2D Mel spectrogram."""
    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=n_mels,
                                          n_fft=N_FFT, hop_length=HOP_LENGTH)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    # Pad or truncate time dimension
    if mel_db.shape[1] < fixed_time:
        mel_db = np.pad(mel_db, ((0, 0), (0, fixed_time - mel_db.shape[1])), mode='constant')
    else:
        mel_db = mel_db[:, :fixed_time]
    return mel_db.astype(np.float32)


def build_mel_dataset(paths, labels, split_name, augment=False, aug_count=1):
    """Build Mel spectrogram dataset for CNN."""
    X_list, y_list = [], []
    for i, (fp, label) in enumerate(zip(paths, labels)):
        if i % 100 == 0:
            print(f'  [{split_name}] {i}/{len(paths)}', end='\r')
        try:
            y_audio = load_and_preprocess(fp)
            mel = extract_mel_spec(y_audio)
            X_list.append(mel)
            y_list.append(label)
            if augment:
                for _ in range(aug_count):
                    y_aug = augment_audio(y_audio, p=0.5)
                    mel_aug = extract_mel_spec(y_aug)
                    X_list.append(mel_aug)
                    y_list.append(label)
        except Exception as e:
            pass

    print(f'  [{split_name}] Done: {len(X_list)} spectrograms          ')
    return np.array(X_list, dtype=np.float32), np.array(y_list)


print('Building CNN Mel spectrogram dataset...')
X_mel_train, y_mel_train = build_mel_dataset(X_train_p, y_train_l, 'train', augment=True, aug_count=1)
X_mel_val, y_mel_val     = build_mel_dataset(X_val_p, y_val_l, 'val')
X_mel_test, y_mel_test   = build_mel_dataset(X_test_p, y_test_l, 'test')

# Encode labels to integers then one-hot
y_mel_train_enc = le.transform(y_mel_train)
y_mel_val_enc   = le.transform(y_mel_val)
y_mel_test_enc  = le.transform(y_mel_test)

y_mel_train_oh = keras.utils.to_categorical(y_mel_train_enc, NUM_CLASSES)
y_mel_val_oh   = keras.utils.to_categorical(y_mel_val_enc, NUM_CLASSES)

# Normalize spectrograms
global_min = X_mel_train.min()
global_max = X_mel_train.max()
X_mel_train_n = (X_mel_train - global_min) / (global_max - global_min + 1e-9)
X_mel_val_n   = (X_mel_val - global_min) / (global_max - global_min + 1e-9)
X_mel_test_n  = (X_mel_test - global_min) / (global_max - global_min + 1e-9)

# Add channel dim
X_mel_train_n = X_mel_train_n[..., np.newaxis]
X_mel_val_n   = X_mel_val_n[..., np.newaxis]
X_mel_test_n  = X_mel_test_n[..., np.newaxis]

print(f'CNN input shape: {X_mel_train_n.shape}')

# Save normalization params
np.save(f'{OUTPUT_DIR}/mel_global_min.npy', np.array([global_min]))
np.save(f'{OUTPUT_DIR}/mel_global_max.npy', np.array([global_max]))

In [ ]:
# ─────────────────────────────────────────────
# Build CNN Architecture
# ─────────────────────────────────────────────
def build_cnn(input_shape, num_classes, dropout=0.4):
    """
    4-block CNN with:
    - Conv2D + BatchNorm + ReLU + MaxPool + Dropout (anti-overfitting)
    - GlobalAveragePooling (instead of Flatten → reduces overfitting)
    - Dense + Dropout + Softmax output
    """
    inputs = keras.Input(shape=input_shape, name='mel_spec')

    # Block 1: 32 filters
    x = layers.Conv2D(32, (3,3), padding='same', kernel_regularizer=regularizers.l2(1e-4))(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D((2,2))(x)
    x = layers.Dropout(0.2)(x)

    # Block 2: 64 filters
    x = layers.Conv2D(64, (3,3), padding='same', kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D((2,2))(x)
    x = layers.Dropout(0.25)(x)

    # Block 3: 128 filters
    x = layers.Conv2D(128, (3,3), padding='same', kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D((2,2))(x)
    x = layers.Dropout(0.3)(x)

    # Block 4: 256 filters
    x = layers.Conv2D(256, (3,3), padding='same', kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.GlobalAveragePooling2D()(x)  # Anti-overfitting: better than Flatten

    # Dense head
    x = layers.Dense(256, kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(dropout)(x)

    x = layers.Dense(128, kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(dropout * 0.5)(x)

    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = keras.Model(inputs, outputs, name='SonicSentinel_CNN')
    return model


input_shape = (X_mel_train_n.shape[1], X_mel_train_n.shape[2], 1)
cnn_model = build_cnn(input_shape, NUM_CLASSES)

# Compile with label smoothing (reduces overconfidence → less overfitting)
cnn_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=['accuracy']
)

cnn_model.summary()

# Class weights for imbalanced data
class_weights = compute_class_weight('balanced',
    classes=np.unique(y_mel_train_enc), y=y_mel_train_enc)
class_weight_dict = dict(zip(np.unique(y_mel_train_enc), class_weights))
print(f'Class weights: {class_weight_dict}')

In [ ]:
# ─────────────────────────────────────────────
# CNN Training Callbacks (overfitting prevention)
# ─────────────────────────────────────────────
cnn_callbacks = [
    # Save best model
    keras.callbacks.ModelCheckpoint(
        filepath=f'{OUTPUT_DIR}/cnn_model.h5',
        monitor='val_accuracy',
        mode='max',
        save_best_only=True,
        verbose=1,
    ),
    # Reduce LR when stuck
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=6,
        min_lr=1e-7,
        verbose=1,
    ),
    # Stop early if no improvement
    keras.callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=18,
        restore_best_weights=True,
        verbose=1,
    ),
    keras.callbacks.TensorBoard(log_dir=f'{OUTPUT_DIR}/tb_logs', histogram_freq=0),
]

print('Starting CNN training...')
print(f'Training samples: {X_mel_train_n.shape[0]}')
print(f'Input shape: {input_shape}')

t0 = time.time()
cnn_history = cnn_model.fit(
    X_mel_train_n, y_mel_train_oh,
    validation_data=(X_mel_val_n, y_mel_val_oh),
    epochs=100,
    batch_size=32,
    callbacks=cnn_callbacks,
    class_weight=class_weight_dict,
    shuffle=True,
    verbose=1,
)
cnn_train_time = time.time() - t0
print(f'\n✓ CNN training complete. Time: {cnn_train_time:.1f}s')

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5), facecolor='#0a0e1a')
for ax in axes: ax.set_facecolor('#0d1526')

hist = cnn_history.history
epochs = range(1, len(hist['accuracy']) + 1)

# Accuracy
axes[0].plot(epochs, hist['accuracy'], '#00d4ff', label='Train', linewidth=2)
axes[0].plot(epochs, hist['val_accuracy'], '#ff6b35', label='Validation', linewidth=2)
axes[0].axhline(0.85, color='#00ff88', linestyle='--', alpha=0.6, label='Target (85%)')
axes[0].set_title('CNN Accuracy', color='#e2e8f0', fontsize=12)
axes[0].set_xlabel('Epoch', color='#94a3b8')
axes[0].set_ylabel('Accuracy', color='#94a3b8')
axes[0].tick_params(colors='#94a3b8')
axes[0].spines[:].set_color('#1e2d4a')
axes[0].legend(facecolor='#0d1526', edgecolor='#1e2d4a', labelcolor='#94a3b8')
axes[0].grid(True, color='#1e2d4a', alpha=0.5)

# Loss
axes[1].plot(epochs, hist['loss'], '#00d4ff', label='Train', linewidth=2)
axes[1].plot(epochs, hist['val_loss'], '#ff6b35', label='Validation', linewidth=2)
axes[1].set_title('CNN Loss', color='#e2e8f0', fontsize=12)
axes[1].set_xlabel('Epoch', color='#94a3b8')
axes[1].set_ylabel('Loss', color='#94a3b8')
axes[1].tick_params(colors='#94a3b8')
axes[1].spines[:].set_color('#1e2d4a')
axes[1].legend(facecolor='#0d1526', edgecolor='#1e2d4a', labelcolor='#94a3b8')
axes[1].grid(True, color='#1e2d4a', alpha=0.5)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/cnn_training_history.png', dpi=120, facecolor='#0a0e1a')
plt.show()

print(f'Final training accuracy: {hist["accuracy"][-1]:.4f}')
print(f'Final validation accuracy: {hist["val_accuracy"][-1]:.4f}')
gap = hist["accuracy"][-1] - hist["val_accuracy"][-1]
print(f'Train-Val gap: {gap:.4f} (< 0.10 = good, no overfitting)')

In [ ]:
# Evaluate CNN
best_cnn = keras.models.load_model(f'{OUTPUT_DIR}/cnn_model.h5')

cnn_proba = best_cnn.predict(X_mel_test_n, verbose=0)
y_pred_cnn = np.argmax(cnn_proba, axis=1)
cnn_acc = accuracy_score(y_mel_test_enc, y_pred_cnn)
cnn_f1  = f1_score(y_mel_test_enc, y_pred_cnn, average='macro', zero_division=0)
cnn_f1w = f1_score(y_mel_test_enc, y_pred_cnn, average='weighted', zero_division=0)

print(f'\nCNN Test Results:')
print(f'  Accuracy:          {cnn_acc:.4f} ({cnn_acc*100:.2f}%)')
print(f'  F1-Score (macro):  {cnn_f1:.4f}')
print(f'  F1-Score (wt):     {cnn_f1w:.4f}')
print(f'\nClassification Report:')
print(classification_report(y_mel_test_enc, y_pred_cnn,
                             target_names=le.classes_, zero_division=0))

# Critical class recall
critical_recalls_cnn = {}
for cls in CRITICAL_CLASSES:
    if cls in le.classes_:
        cls_idx = list(le.classes_).index(cls)
        rec = recall_score((y_mel_test_enc == cls_idx).astype(int),
                           (y_pred_cnn == cls_idx).astype(int), zero_division=0)
        critical_recalls_cnn[cls] = rec
        status = '✓' if rec >= 0.85 else '✗'
        print(f'{status} {CLASS_LABELS.get(cls, cls)}: {rec:.4f}')

## 📈 Step 12: Model Comparison & Selection

In [ ]:
# ─────────────────────────────────────────────
# Comprehensive Model Comparison
# ─────────────────────────────────────────────
print('='*70)
print('MODEL COMPARISON SUMMARY')
print('='*70)

model_results = {
    'SVM': {'accuracy': svm_acc, 'f1_macro': svm_f1, 'f1_weighted': svm_f1w,
            'critical_recalls': critical_recalls_svm, 'training_time': svm_train_time},
    'Random Forest': {'accuracy': rf_acc, 'f1_macro': rf_f1, 'f1_weighted': rf_f1w,
                      'critical_recalls': critical_recalls_rf, 'training_time': rf_train_time},
    'CNN': {'accuracy': cnn_acc, 'f1_macro': cnn_f1, 'f1_weighted': cnn_f1w,
            'critical_recalls': critical_recalls_cnn, 'training_time': cnn_train_time},
}

print(f'{"Model":<20} {"Accuracy":>10} {"F1-Macro":>10} {"F1-Wtd":>10} {"Avg Crit Rec":>14}')
print('-'*70)

best_model_name = None
best_score = -1

for name, res in model_results.items():
    avg_crit = np.mean(list(res['critical_recalls'].values())) if res['critical_recalls'] else 0
    # Composite score: 35% accuracy + 45% f1_macro + 20% critical recall
    composite = res['accuracy'] * 0.35 + res['f1_macro'] * 0.45 + avg_crit * 0.20
    print(f'{name:<20} {res["accuracy"]:>10.4f} {res["f1_macro"]:>10.4f} {res["f1_weighted"]:>10.4f} {avg_crit:>14.4f}')
    if composite > best_score:
        best_score = composite
        best_model_name = name

print(f'\n🏆 Best Model: {best_model_name} (composite score: {best_score:.4f})')

# SRS Requirements Check
print('\n── SRS Requirements Check ──')
for name, res in model_results.items():
    acc_ok = '✓' if res['accuracy'] >= 0.85 else '✗'
    f1_ok  = '✓' if res['f1_macro'] >= 0.80 else '✗'
    avg_crit = np.mean(list(res['critical_recalls'].values())) if res['critical_recalls'] else 0
    crit_ok = '✓' if avg_crit >= 0.85 else '✗'
    print(f'  {name}: Accuracy {acc_ok} ({res["accuracy"]*100:.1f}%) | '
          f'F1 {f1_ok} ({res["f1_macro"]:.3f}) | CritRec {crit_ok} ({avg_crit:.3f})')

In [ ]:
# Visualize model comparison
models_list = list(model_results.keys())
metrics_names = ['accuracy', 'f1_macro', 'f1_weighted']
colors_list = ['#00d4ff', '#ff6b35', '#a855f7']

fig, axes = plt.subplots(1, 3, figsize=(16, 5), facecolor='#0a0e1a')
for ax in axes: ax.set_facecolor('#0d1526')

x = np.arange(len(models_list))
for i, (metric, ax) in enumerate(zip(metrics_names, axes)):
    vals = [model_results[m][metric] for m in models_list]
    bars = ax.bar(x, vals, color=colors_list, edgecolor='none', width=0.5)
    ax.set_title(metric.replace('_', ' ').title(), color='#e2e8f0', fontsize=11)
    ax.set_xticks(x)
    ax.set_xticklabels(models_list, color='#94a3b8', rotation=15)
    ax.set_ylim([0, 1.1])
    ax.axhline(0.85 if metric == 'accuracy' else 0.80, color='#00ff88',
               linestyle='--', alpha=0.7, label='Target')
    ax.tick_params(colors='#94a3b8')
    ax.spines[:].set_color('#1e2d4a')
    ax.grid(axis='y', color='#1e2d4a', alpha=0.5)
    ax.legend(facecolor='#0d1526', edgecolor='#1e2d4a', labelcolor='#94a3b8', fontsize=8)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, val + 0.01,
                f'{val:.3f}', ha='center', va='bottom', color='#e2e8f0', fontsize=9)

plt.suptitle('Model Comparison: SVM vs Random Forest vs CNN', color='#e2e8f0', fontsize=13)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/model_comparison.png', dpi=120, facecolor='#0a0e1a')
plt.show()

## 🔲 Step 13: Confusion Matrices

In [ ]:
def plot_confusion_matrix(cm, class_names, title, save_path):
    """Plot normalized confusion matrix."""
    cm_norm = cm.astype('float') / (cm.sum(axis=1, keepdims=True) + 1e-9)
    short_names = [n.replace('_', '\n') for n in class_names]

    fig, ax = plt.subplots(figsize=(14, 11), facecolor='#0a0e1a')
    ax.set_facecolor('#0a0e1a')

    im = ax.imshow(cm_norm, cmap='Blues', aspect='auto', vmin=0, vmax=1)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04).ax.yaxis.set_tick_params(color='#94a3b8')

    # Annotate cells
    thresh = cm_norm.max() / 2.0
    for i in range(len(class_names)):
        for j in range(len(class_names)):
            color = 'white' if cm_norm[i,j] < thresh else 'black'
            ax.text(j, i, f'{cm_norm[i,j]:.2f}\n({cm[i,j]})',
                    ha='center', va='center', color=color, fontsize=7.5)

    ax.set_xticks(range(len(class_names)))
    ax.set_yticks(range(len(class_names)))
    ax.set_xticklabels(short_names, color='#94a3b8', rotation=45, ha='right', fontsize=9)
    ax.set_yticklabels(short_names, color='#94a3b8', fontsize=9)
    ax.set_xlabel('Predicted Label', color='#94a3b8', fontsize=11)
    ax.set_ylabel('True Label', color='#94a3b8', fontsize=11)
    ax.set_title(title, color='#e2e8f0', fontsize=13, pad=15)

    plt.tight_layout()
    plt.savefig(save_path, dpi=130, facecolor='#0a0e1a', bbox_inches='tight')
    plt.show()


# SVM Confusion Matrix
cm_svm = confusion_matrix(y_test_enc, y_pred_svm)
plot_confusion_matrix(cm_svm, list(le.classes_),
                       'Confusion Matrix — SVM',
                       f'{OUTPUT_DIR}/cm_svm.png')

# RF Confusion Matrix
cm_rf = confusion_matrix(y_test_enc, y_pred_rf)
plot_confusion_matrix(cm_rf, list(le.classes_),
                       'Confusion Matrix — Random Forest',
                       f'{OUTPUT_DIR}/cm_rf.png')

# CNN Confusion Matrix
cm_cnn = confusion_matrix(y_mel_test_enc, y_pred_cnn)
plot_confusion_matrix(cm_cnn, list(le.classes_),
                       'Confusion Matrix — CNN',
                       f'{OUTPUT_DIR}/cm_cnn.png')

print('✓ Confusion matrices saved')

## 💾 Step 14: Save Best Model & Artifacts

In [ ]:
# Select and save the best sklearn model as 'best_model.pkl'
best_sklearn_models = {'SVM': best_svm, 'Random Forest': best_rf}
best_sklearn_name = max(['SVM', 'Random Forest'],
                        key=lambda n: model_results[n]['f1_macro'])
best_sklearn = best_sklearn_models[best_sklearn_name]

with open(f'{OUTPUT_DIR}/best_model.pkl', 'wb') as f:
    pickle.dump(best_sklearn, f)

print(f'Best sklearn model: {best_sklearn_name}')

# Save label encoder and scaler (already saved, confirm)
with open(f'{OUTPUT_DIR}/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
with open(f'{OUTPUT_DIR}/label_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)

# Save model metadata
metadata = {
    'best_sklearn_model': best_sklearn_name,
    'sound_classes': list(le.classes_),
    'num_classes': NUM_CLASSES,
    'sample_rate': SAMPLE_RATE,
    'n_mfcc': N_MFCC,
    'n_mels': N_MELS,
    'hop_length': HOP_LENGTH,
    'n_fft': N_FFT,
    'segment_duration': SEGMENT_DUR,
    'model_version': '1.0.0',
    'mel_global_min': float(global_min),
    'mel_global_max': float(global_max),
    'results': {
        name: {
            'accuracy': res['accuracy'],
            'f1_macro': res['f1_macro'],
            'f1_weighted': res['f1_weighted'],
        } for name, res in model_results.items()
    }
}

with open(f'{OUTPUT_DIR}/model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print('✓ All models and artifacts saved')
print(f'Output directory: {OUTPUT_DIR}')
!ls -lh {OUTPUT_DIR}

## 🔬 Step 15: Noise Robustness Testing

In [ ]:
# Test model robustness against noisy inputs
print('Noise Robustness Analysis...')

noise_levels = [0.0, 0.01, 0.05, 0.1, 0.2]
noise_results_svm = []
noise_results_rf  = []
noise_results_cnn = []

# Use a subset of test files for speed
test_subset_size = min(100, len(X_test_p))
test_subset_idx = np.random.choice(len(X_test_p), test_subset_size, replace=False)
test_paths_sub = X_test_p[test_subset_idx]
test_labels_sub = y_test_l[test_subset_idx]

for noise_level in noise_levels:
    X_noisy, y_noisy = [], []
    X_mel_noisy = []

    for fp, label in zip(test_paths_sub, test_labels_sub):
        try:
            y_audio = load_and_preprocess(fp)
            # Add noise
            if noise_level > 0:
                noise = np.random.randn(len(y_audio)).astype(np.float32) * noise_level
                y_audio = np.clip(y_audio + noise, -1.0, 1.0)
            feats = extract_features(y_audio)
            X_noisy.append(feats)
            y_noisy.append(label)
            mel = extract_mel_spec(y_audio)
            mel_n = (mel - global_min) / (global_max - global_min + 1e-9)
            X_mel_noisy.append(mel_n[np.newaxis, ..., np.newaxis])
        except Exception:
            pass

    if not X_noisy:
        continue

    X_noisy_arr = np.array(X_noisy, dtype=np.float32)
    y_noisy_enc = le.transform(y_noisy)
    X_noisy_sc  = scaler.transform(X_noisy_arr)

    acc_svm = accuracy_score(y_noisy_enc, best_svm.predict(X_noisy_sc))
    acc_rf  = accuracy_score(y_noisy_enc, best_rf.predict(X_noisy_sc))

    X_mel_arr = np.concatenate(X_mel_noisy, axis=0)
    acc_cnn = accuracy_score(y_noisy_enc, np.argmax(best_cnn.predict(X_mel_arr, verbose=0), axis=1))

    noise_results_svm.append(acc_svm)
    noise_results_rf.append(acc_rf)
    noise_results_cnn.append(acc_cnn)
    print(f'  Noise {noise_level:.2f}: SVM={acc_svm:.3f} | RF={acc_rf:.3f} | CNN={acc_cnn:.3f}')

# Plot robustness
fig, ax = plt.subplots(figsize=(10, 5), facecolor='#0a0e1a')
ax.set_facecolor('#0d1526')
ax.plot(noise_levels, noise_results_svm, '#00d4ff', marker='o', label='SVM', linewidth=2)
ax.plot(noise_levels, noise_results_rf,  '#ff6b35', marker='s', label='RF', linewidth=2)
ax.plot(noise_levels, noise_results_cnn, '#a855f7', marker='^', label='CNN', linewidth=2)
ax.axhline(0.85, color='#00ff88', linestyle='--', alpha=0.6, label='Target')
ax.set_xlabel('Noise Level (σ)', color='#94a3b8')
ax.set_ylabel('Accuracy', color='#94a3b8')
ax.set_title('Noise Robustness Analysis', color='#e2e8f0', fontsize=12)
ax.tick_params(colors='#94a3b8')
ax.spines[:].set_color('#1e2d4a')
ax.legend(facecolor='#0d1526', edgecolor='#1e2d4a', labelcolor='#94a3b8')
ax.grid(True, color='#1e2d4a', alpha=0.5)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/noise_robustness.png', dpi=120, facecolor='#0a0e1a')
plt.show()

## 📋 Step 16: Final Summary & Export

In [ ]:
print('\n' + '='*70)
print('SONICSENTINEL AI — TRAINING COMPLETE SUMMARY')
print('='*70)

for name, res in model_results.items():
    avg_crit = np.mean(list(res['critical_recalls'].values())) if res['critical_recalls'] else 0
    acc_ok   = '✓' if res['accuracy'] >= 0.85 else '✗'
    f1_ok    = '✓' if res['f1_macro'] >= 0.80 else '✗'
    crit_ok  = '✓' if avg_crit >= 0.85 else '✗'

    print(f'\n── {name} ──')
    print(f'  Accuracy:           {res["accuracy"]*100:.2f}%  {acc_ok} (target ≥ 85%)')
    print(f'  F1-Score (macro):   {res["f1_macro"]:.4f}   {f1_ok} (target ≥ 0.80)')
    print(f'  Avg Critical Rec:   {avg_crit:.4f}   {crit_ok} (target ≥ 0.85)')
    print(f'  Training Time:      {res["training_time"]:.1f}s')

print(f'\n🏆 Best Overall Model: {best_model_name}')

# Copy best model to Drive for use in backend
import shutil
drive_output = '/content/drive/MyDrive/SonicSentinel/models'
Path(drive_output).mkdir(parents=True, exist_ok=True)

for fname in ['best_model.pkl', 'scaler.pkl', 'label_encoder.pkl',
              'cnn_model.h5', 'model_metadata.json',
              'X_test.npy', 'y_test.npy']:
    src = f'{OUTPUT_DIR}/{fname}'
    dst = f'{drive_output}/{fname}'
    if Path(src).exists():
        shutil.copy2(src, dst)
        print(f'  ✓ Saved: {fname}')

# Also copy confusion matrix images and plots
for fname in Path(OUTPUT_DIR).glob('*.png'):
    shutil.copy2(str(fname), f'{drive_output}/{fname.name}')

print(f'\n✓ All artifacts saved to Google Drive: {drive_output}')
print('\nReady for backend integration!')

In [ ]:
# Quick inference test
print('=== Quick Inference Test ===')

test_sample = df_manifest[df_manifest['class_label'] == 'gunshot'].iloc[0]
y_infer = load_and_preprocess(test_sample['filepath'])
feats_infer = extract_features(y_infer)
feats_scaled = scaler.transform(feats_infer.reshape(1, -1))

# Sklearn prediction
pred_enc = best_sklearn.predict(feats_scaled)
pred_class = le.inverse_transform(pred_enc)[0]
proba = best_sklearn.predict_proba(feats_scaled)[0]

confidence_scores = {cls: float(proba[i]) for i, cls in enumerate(le.classes_)}
sorted_scores = sorted(confidence_scores.items(), key=lambda x: x[1], reverse=True)

print(f'File: {test_sample["filename"]}')
print(f'Actual class: {test_sample["class_label"]}')
print(f'Predicted class: {pred_class}')
print(f'Confidence: {proba[pred_enc[0]]:.4f}')
print(f'Correct: {pred_class == test_sample["class_label"]}')
print(f'\nTop-3 predictions:')
for cls, score in sorted_scores[:3]:
    bar = '█' * int(score * 30)
    print(f'  {CLASS_LABELS.get(cls, cls):<28} {bar:<30} {score:.4f}')

# CNN prediction
mel_infer = extract_mel_spec(y_infer)
mel_infer_n = (mel_infer - global_min) / (global_max - global_min + 1e-9)
mel_infer_n = mel_infer_n[np.newaxis, ..., np.newaxis]
cnn_proba_infer = best_cnn.predict(mel_infer_n, verbose=0)[0]
cnn_pred_idx = np.argmax(cnn_proba_infer)
cnn_pred_class = le.classes_[cnn_pred_idx]

print(f'\nCNN Prediction:')
print(f'  Predicted: {cnn_pred_class} ({cnn_proba_infer[cnn_pred_idx]:.4f})')
print(f'  Match with sklearn: {cnn_pred_class == pred_class}')

print('\n✓ Inference test complete!')